#Amazon review

In [6]:
cd /home/m00m53/data/Rawdata/

/home/m00m53/data/Rawdata


In [ ]:
import gzip
from collections import defaultdict
from datetime import datetime
import json

def parse(path):
    g = gzip.open(path, 'r')
    for l in g:
        yield json.loads(l)

def convert_time(unix_time):
    return datetime.utcfromtimestamp(unix_time).strftime('%Y-%m-%d')

countU = defaultdict(lambda: 0)
countP = defaultdict(lambda: 0)
line = 0

dataset_name = 'Videogame' #Beauty, Music, Videogame

# 1차 패스: 각 사용자 및 아이템별 리뷰 개수 카운트
for l in parse('reviews_' + dataset_name + '.json.gz'):
    line += 1
    asin = l['asin']
    rev = l['reviewerID']
    countU[rev] += 1
    countP[asin] += 1

usermap = dict()
usernum = 0
itemmap = dict()
itemnum = 0
User = dict()


for l in parse('reviews_' + dataset_name + '.json.gz'):
    asin = l['asin']
    rev = l['reviewerID']
    time = l['unixReviewTime']
    overall = l['overall']
    
    # 5번 미만 리뷰 작성자/대상 아이템 필터링
    if countU[rev] < 5 or countP[asin] < 5:
        continue

    # 사용자 ID 매핑 및 10,000명 제한 로직
    if rev in usermap:
        userid = usermap[rev]
    else:
        # Videogame 데이터셋인 경우 10,000명이 넘어가면 새로운 사용자를 추가하지 않음
        if dataset_name == 'Videogame' and usernum >= 10000:
            continue
        usernum += 1
        userid = usernum
        usermap[rev] = userid
        User[userid] = []

    # 아이템 ID 매핑
    if asin in itemmap:
        itemid = itemmap[asin]
    else:
        itemnum += 1
        itemid = itemnum
        itemmap[asin] = itemid

    User[userid].append([time, itemid, overall])

# 각 사용자별로 시간순 정렬
for userid in User.keys():
    User[userid].sort(key=lambda x: x[0])

print(f"Total Users: {usernum}, Total Items: {itemnum}")

# 1. 중복 제거 및 시간 포함 데이터 저장
unique_time = set()
for user in User.keys():
    for i in User[user]:
        unique_time.add('%d %d %f %s' % (user, i[1], i[2], convert_time(i[0])))

sorted_unique_time = sorted(unique_time, key=lambda x: int(x.split()[0]))

# 파일명 예시: Videogameduple_1_10000.txt
output_name = dataset_name + 'duple'
if dataset_name == 'Videogame':
    output_name += '_1_10000'

with open(output_name + '.txt', 'w') as f_time:
    for line in sorted_unique_time:
        f_time.write(line + '\n')

# 2. 시간 제외 (notime) 데이터 저장
unique_notime = set()
for user in User.keys():
    for i in User[user]:
        unique_notime.add('%d %d %f' % (user, i[1], i[2]))

sorted_unique_notime = sorted(unique_notime, key=lambda x: int(x.split()[0]))

with open(dataset_name + '_notime_duple_test.txt', 'w') as f_notime:
    for line in sorted_unique_notime:
        f_notime.write(line + '\n')

# 3. ID만 포함 (onlyid) 데이터 저장
unique_onlyid = set()
for user in User.keys():
    for i in User[user]:
        unique_onlyid.add('%d %d' % (user, i[1]))

sorted_unique_onlyid = sorted(unique_onlyid, key=lambda x: int(x.split()[0]))

with open(dataset_name + '_onlyid_duple_test.txt', 'w') as f_onlyid:
    for line in sorted_unique_onlyid:
        f_onlyid.write(line + '\n')

#Movielens

In [ ]:
import pandas as pd

# rating.dat 파일을 읽어서 데이터프레임으로 변환
columns = ['UserID', 'MovieID', 'Rating', 'Timestamp']
rating_df = pd.read_csv('/content/ratings.dat', sep='::', header=None, names=columns, engine='python')
rating_df_onlyid = rating_df.iloc[:, [0, 1]]
# 데이터프레임 확인
print(rating_df.head())

# Timestamp 열을 유닉스 타임에서 Pandas datetime 객체로 변환
rating_df['Timestamp'] = pd.to_datetime(rating_df['Timestamp'], unit='s')

# 원하는 형식으로 포맷팅 (예: 'YYYY-MM-DD')
rating_df['Timestamp'] = rating_df['Timestamp'].dt.strftime('%Y-%m-%d')

# 변환된 데이터프레임 확인
print(rating_df.head())

# 데이터프레임을 스페이스바로 구분된 텍스트 파일로 저장
rating_df.to_csv('Movielens_duple.txt', sep=' ', index=False, header=False)